<a href="https://colab.research.google.com/github/kligeys/AI-Projects/blob/main/MCP_server_client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import socket
import threading

HOST = '127.0.0.1'  # Standard loopback interface address (localhost)
PORT = 65432        # Port to listen on (non-privileged ports are > 1023)

def call_calculator_server(expression):
    """Evaluates a simple arithmetic expression."""
    try:
        # Using eval() can be dangerous; for a real application, a more robust
        # and secure parsing approach would be needed.
        # This example assumes trusted input for simplicity.
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def handle_client(conn, addr):
    """Handles a single client connection."""
    print(f"Connected by {addr}")
    with conn:
        while True:
            data = conn.recv(1024) # Receive up to 1024 bytes
            if not data: # Client disconnected
                break
            request = data.decode('utf-8').strip()
            print(f"Received from {addr}: {request}")

            # Process the request (assuming it's an arithmetic expression)
            response = calculate(request)

            conn.sendall(response.encode('utf-8'))
    print(f"Disconnected from {addr}")

def start_server():
    """Starts the MCP calculator server."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((HOST, PORT))
        s.listen()
        print(f"MCP Calculator Server listening on {HOST}:{PORT}")
        while True:
            conn, addr = s.accept()
            # Handle client connection in a new thread
            client_thread = threading.Thread(target=handle_client, args=(conn, addr))
            client_thread.start()

# Note: Running this cell will start the server indefinitely.
# To stop it, you may need to interrupt the kernel or restart the runtime.


To run the server, execute the cell below. It will listen for incoming connections. You can then connect to it using a client (which you'd need to create separately, for example, using `socket` in another Python script or even `netcat` from a terminal).

In [ ]:
import subprocess
import os
import signal

# Find the Process ID (PID) using port 65432
try:
    # lsof -t -i :65432 gets the PID of the process using that port
    pid = subprocess.check_output(["lsof", "-t", "-i", ":65432"]).decode('utf-8').strip()
    if pid:
        pids = [int(p) for p in pid.split()]
        for p in pids:
            print(f"Killing background server process with PID: {p}...")
            os.kill(p, signal.SIGKILL)
        print("Port 65432 successfully freed!")
    else:
        print("No process found holding port 65432.")
except subprocess.CalledProcessError:
    print("Port 65432 is already free (no active process found).")

In [1]:
import socket
import threading

HOST = '127.0.0.1'
PORT = 65432

def calculate(expression):
    """Evaluates a simple arithmetic expression safely."""
    try:
        # Using eval here for demonstration purposes
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def handle_client(conn, addr):
    """Handles a single client connection."""
    print(f"Connected by {addr}")
    with conn:
        while True:
            data = conn.recv(1024)
            if not data:
                break
            request = data.decode('utf-8').strip()
            print(f"Received from {addr}: {request}")

            response = calculate(request)
            conn.sendall(response.encode('utf-8'))
    print(f"Disconnected from {addr}")

def start_server():
    """Starts the MCP calculator server using SO_REUSEADDR to avoid address conflicts."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORT))
        s.listen()
        print(f"MCP Calculator Server listening on {HOST}:{PORT}")
        while True:
            try:
                conn, addr = s.accept()
                client_thread = threading.Thread(target=handle_client, args=(conn, addr))
                client_thread.daemon = True
                client_thread.start()
            except Exception as e:
                print(f"Server loop encounter error: {e}")
                break

# Start server in background thread
server_thread = threading.Thread(target=start_server)
server_thread.daemon = True
server_thread.start()
print("Server restarted and listening on port 65432!")

Server restarted and listening on port 65432!
MCP Calculator Server listening on 127.0.0.1:65432


### LLM Agent Client

This section uses the Gemini API. Make sure you have added your Gemini API key to the Colab Secrets manager (the key icon on the left panel) with the name `GOOGLE_API_KEY`.

In [ ]:
!pip install openai -q

### OpenAI ChatGPT Agent Client

This section uses the OpenAI API. Ensure you have added your OpenAI API key to the Colab Secrets manager with the name `OPENAI_API_KEY`.

In [2]:
import os
import json
import socket
from google.colab import userdata
from openai import OpenAI

def call_calculator_server(expression):
    """Establishes a socket connection to the local background server and evaluates the expression."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.connect(('127.0.0.1', 65432))
            s.sendall(expression.encode('utf-8'))
            data = s.recv(1024)
            return data.decode('utf-8').strip()
    except Exception as e:
        return f"Client error connecting to server: {e}"

try:
    # Initialize OpenAI client with key from Colab Secrets
    openai_api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=openai_api_key)

    # Define the tool schema for OpenAI
    tools = [
        {
            "type": "function",
            "function": {
                "name": "calculate",
                "description": "Sends an arithmetic expression to the local calculator server and returns the result.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "A valid Python arithmetic expression (e.g., '2 + 2', '10 * 5 / 2')"
                        }
                    },
                    "required": ["expression"]
                }
            }
        }
    ]

    user_query = "What is 1000 divided by 3, and then add 100 to the result?"
    print(f"User Query: {user_query}\n")

    messages = [{"role": "user", "content": user_query}]

    # Initial request to the model
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    # Check if the model wants to call our tool
    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            if tool_call.function.name == "calculate":
                # Extract the arguments the model generated
                function_args = json.loads(tool_call.function.arguments)
                expression = function_args.get("expression")

                print(f"[Agent Action] Decided to calculate: {expression}")

                # Call our previously defined python function that talks to the server
                function_response = call_calculator_server(expression)
                print(f"[Server Output] {function_response}")

                # Append the function response back to the messages list
                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": "calculate",
                        "content": str(function_response),
                    }
                )

        # Send the tool result back to the model for the final answer
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        print(f"\nAgent Final Response: {second_response.choices[0].message.content}")
    else:
        print(f"\nAgent Response: {response_message.content}")

except userdata.SecretNotFoundError:
    print("Please add your OPENAI_API_KEY to the Colab secrets manager.")
except Exception as e:
    print(f"An error occurred: {e}")

User Query: What is 1000 divided by 3, and then add 100 to the result?

[Agent Action] Decided to calculate: 1000 / 3 + 100
Connected by ('127.0.0.1', 57082)
Received from ('127.0.0.1', 57082): 1000 / 3 + 100
[Server Output] 433.3333333333333
Disconnected from ('127.0.0.1', 57082)

Agent Final Response: 1000 divided by 3 is approximately 333.33. When you add 100 to that result, you get approximately 433.33.
